In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

In [2]:
BASE_PATH = Path(
    r"C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI"
)

CLASSIFICATION_PATH = (
    BASE_PATH / "models" / "final"
)

REGRESSION_PATH = (
    BASE_PATH / "models" / "regression"
)

classification_model = joblib.load(
    CLASSIFICATION_PATH /
    "emi_eligibility_classifier.pkl"
)

classification_preprocessor = joblib.load(
    CLASSIFICATION_PATH /
    "classification_preprocessor.pkl"
)

regression_model = joblib.load(
    REGRESSION_PATH /
    "emi_amount_regressor.pkl"
)

regression_preprocessor = joblib.load(
    REGRESSION_PATH /
    "emi_regression_preprocessor.pkl"
)

print("All final models loaded successfully!")

All final models loaded successfully!


In [3]:
DATA_PATH = (
    BASE_PATH /
    "data" /
    "processed" /
    "emi_prediction_dataset_cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (404800, 27)


In [4]:
expense_columns = [
    "monthly_rent",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses"
]

df["total_living_expenses"] = (
    df[expense_columns].sum(axis=1)
)

df["total_monthly_commitments"] = (
    df["total_living_expenses"]
    + df["current_emi_amount"]
)

df["disposable_income"] = (
    df["monthly_salary"]
    - df["total_monthly_commitments"]
)

salary_safe = (
    df["monthly_salary"].replace(0, np.nan)
)

df["expense_to_income_ratio"] = (
    df["total_living_expenses"]
    / salary_safe
)

df["commitment_to_income_ratio"] = (
    df["total_monthly_commitments"]
    / salary_safe
)

df["current_emi_to_income_ratio"] = (
    df["current_emi_amount"]
    / salary_safe
)

df["requested_amount_to_income"] = (
    df["requested_amount"]
    / salary_safe
)

df["emergency_fund_to_income"] = (
    df["emergency_fund"]
    / salary_safe
)

print("Feature engineering completed!")

Feature engineering completed!


In [5]:
classification_features = df.drop(
    columns=[
        "emi_eligibility",
        "max_monthly_emi"
    ]
)

actual_classification = df[
    "emi_eligibility"
]

X_classification_processed = (
    classification_preprocessor.transform(
        classification_features
    )
)

classification_predictions = (
    classification_model.predict(
        X_classification_processed
    )
)

print("Classification predictions generated!")

print(
    "\nPrediction distribution:"
)

print(
    pd.Series(
        classification_predictions
    ).value_counts()
)

Classification predictions generated!

Prediction distribution:
Not_Eligible    257062
High_Risk        80410
Eligible         67328
Name: count, dtype: int64


In [6]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report
)

print("=" * 60)
print("FINAL CLASSIFICATION VALIDATION")
print("=" * 60)

print(
    "Accuracy:",
    accuracy_score(
        actual_classification,
        classification_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        actual_classification,
        classification_predictions
    )
)

print("\nClassification Report:")

print(
    classification_report(
        actual_classification,
        classification_predictions,
        digits=4,
        zero_division=0
    )
)

FINAL CLASSIFICATION VALIDATION
Accuracy: 0.794824604743083
Balanced Accuracy: 0.7501864446308845

Classification Report:
              precision    recall  f1-score   support

    Eligible     0.8369    0.7569    0.7949     74444
   High_Risk     0.1487    0.6836    0.2442     17488
Not_Eligible     0.9859    0.8101    0.8894    312868

    accuracy                         0.7948    404800
   macro avg     0.6572    0.7502    0.6428    404800
weighted avg     0.9223    0.7948    0.8441    404800



In [7]:
regression_features = df.drop(
    columns=[
        "emi_eligibility",
        "max_monthly_emi"
    ]
)

actual_emi = df[
    "max_monthly_emi"
]

X_regression_processed = (
    regression_preprocessor.transform(
        regression_features
    )
)

regression_predictions = (
    regression_model.predict(
        X_regression_processed
    )
)

print("Regression predictions generated!")

print(
    "Minimum prediction:",
    regression_predictions.min()
)

print(
    "Maximum prediction:",
    regression_predictions.max()
)

Regression predictions generated!
Minimum prediction: -3047.205962335316
Maximum prediction: 84669.20262841442


In [8]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

validation_mae = mean_absolute_error(
    actual_emi,
    regression_predictions
)

validation_rmse = np.sqrt(
    mean_squared_error(
        actual_emi,
        regression_predictions
    )
)

validation_r2 = r2_score(
    actual_emi,
    regression_predictions
)

print("=" * 60)
print("FINAL REGRESSION VALIDATION")
print("=" * 60)

print(
    f"MAE : ₹{validation_mae:,.2f}"
)

print(
    f"RMSE: ₹{validation_rmse:,.2f}"
)

print(
    f"R²  : {validation_r2:.4f}"
)

FINAL REGRESSION VALIDATION
MAE : ₹361.00
RMSE: ₹801.74
R²  : 0.9893


In [9]:
validation_results = pd.DataFrame({
    "Actual_Eligibility": actual_classification.values,
    "Predicted_Eligibility": classification_predictions,
    "Actual_Max_EMI": actual_emi.values,
    "Predicted_Max_EMI": regression_predictions
})

display(
    validation_results.head(20)
)

,Actual_Eligibility,Predicted_Eligibility,Actual_Max_EMI,Predicted_Max_EMI
0,Not_Eligible,Not_Eligible,500.00,473.731280
1,Not_Eligible,Not_Eligible,700.00,712.642293
2,Eligible,High_Risk,27775.00,27770.305756
3,Eligible,Eligible,16170.00,15508.804748
4,Not_Eligible,Not_Eligible,500.00,664.176681
5,Not_Eligible,Not_Eligible,500.00,648.361062
6,Not_Eligible,Not_Eligible,1950.00,1790.631403
7,Not_Eligible,High_Risk,8260.00,4670.474706
8,Not_Eligible,High_Risk,5500.00,5585.921242
9,Not_Eligible,Not_Eligible,9355.50,9287.194828


In [10]:
validation_results["EMI_Error"] = (
    validation_results["Actual_Max_EMI"]
    - validation_results["Predicted_Max_EMI"]
)

validation_results["Absolute_EMI_Error"] = (
    validation_results["EMI_Error"].abs()
)

display(
    validation_results.head(10)
)

,Actual_Eligibility,Predicted_Eligibility,Actual_Max_EMI,Predicted_Max_EMI,EMI_Error,Absolute_EMI_Error
0,Not_Eligible,Not_Eligible,500.0,473.731280,26.268720,26.268720
1,Not_Eligible,Not_Eligible,700.0,712.642293,-12.642293,12.642293
2,Eligible,High_Risk,27775.0,27770.305756,4.694244,4.694244
3,Eligible,Eligible,16170.0,15508.804748,661.195252,661.195252
4,Not_Eligible,Not_Eligible,500.0,664.176681,-164.176681,164.176681
5,Not_Eligible,Not_Eligible,500.0,648.361062,-148.361062,148.361062
6,Not_Eligible,Not_Eligible,1950.0,1790.631403,159.368597,159.368597
7,Not_Eligible,High_Risk,8260.0,4670.474706,3589.525294,3589.525294
8,Not_Eligible,High_Risk,5500.0,5585.921242,-85.921242,85.921242
9,Not_Eligible,Not_Eligible,9355.5,9287.194828,68.305172,68.305172


In [11]:
OUTPUT_PATH = (
    BASE_PATH /
    "data" /
    "processed"
)

validation_file = (
    OUTPUT_PATH /
    "model_validation_results.csv"
)

validation_results.to_csv(
    validation_file,
    index=False
)

print("Validation results saved!")
print("Path:", validation_file)

Validation results saved!
Path: C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\data\processed\model_validation_results.csv


In [12]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [13]:
BASE_PATH = Path(
    r"C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI"
)

DATA_PATH = (
    BASE_PATH
    / "data"
    / "processed"
    / "emi_prediction_dataset_cleaned.csv"
)

CLASSIFICATION_PATH = (
    BASE_PATH
    / "models"
    / "final"
)

REGRESSION_PATH = (
    BASE_PATH
    / "models"
    / "regression"
)

print("Project paths configured successfully!")

Project paths configured successfully!


In [14]:
classification_model = joblib.load(
    CLASSIFICATION_PATH /
    "emi_eligibility_classifier.pkl"
)

classification_preprocessor = joblib.load(
    CLASSIFICATION_PATH /
    "classification_preprocessor.pkl"
)

regression_model = joblib.load(
    REGRESSION_PATH /
    "emi_amount_regressor.pkl"
)

regression_preprocessor = joblib.load(
    REGRESSION_PATH /
    "emi_regression_preprocessor.pkl"
)

print("All final models loaded successfully!")

All final models loaded successfully!


In [15]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (404800, 27)


In [16]:
expense_columns = [
    "monthly_rent",
    "school_fees",
    "college_fees",
    "travel_expenses",
    "groceries_utilities",
    "other_monthly_expenses"
]

df["total_living_expenses"] = (
    df[expense_columns].sum(axis=1)
)

df["total_monthly_commitments"] = (
    df["total_living_expenses"]
    + df["current_emi_amount"]
)

df["disposable_income"] = (
    df["monthly_salary"]
    - df["total_monthly_commitments"]
)

salary_safe = (
    df["monthly_salary"].replace(0, np.nan)
)

df["expense_to_income_ratio"] = (
    df["total_living_expenses"]
    / salary_safe
)

df["commitment_to_income_ratio"] = (
    df["total_monthly_commitments"]
    / salary_safe
)

df["current_emi_to_income_ratio"] = (
    df["current_emi_amount"]
    / salary_safe
)

df["requested_amount_to_income"] = (
    df["requested_amount"]
    / salary_safe
)

df["emergency_fund_to_income"] = (
    df["emergency_fund"]
    / salary_safe
)

print("Financial features recreated successfully!")

Financial features recreated successfully!


In [17]:
X_classification = df.drop(
    columns=[
        "emi_eligibility",
        "max_monthly_emi"
    ]
)

y_classification = df[
    "emi_eligibility"
]

print("Classification X shape:", X_classification.shape)
print("Classification y shape:", y_classification.shape)

Classification X shape: (404800, 33)
Classification y shape: (404800,)


In [18]:
X_classification_processed = (
    classification_preprocessor.transform(
        X_classification
    )
)

classification_predictions = (
    classification_model.predict(
        X_classification_processed
    )
)

print("Classification predictions generated!")
print(
    "Number of predictions:",
    len(classification_predictions)
)

Classification predictions generated!
Number of predictions: 404800


In [19]:
classification_accuracy = accuracy_score(
    y_classification,
    classification_predictions
)

classification_balanced_accuracy = (
    balanced_accuracy_score(
        y_classification,
        classification_predictions
    )
)

print("=" * 60)
print("FINAL CLASSIFICATION VALIDATION")
print("=" * 60)

print(
    f"Accuracy           : "
    f"{classification_accuracy:.4f}"
)

print(
    f"Balanced Accuracy  : "
    f"{classification_balanced_accuracy:.4f}"
)

print("\nClassification Report:")

print(
    classification_report(
        y_classification,
        classification_predictions,
        digits=4,
        zero_division=0
    )
)

FINAL CLASSIFICATION VALIDATION
Accuracy           : 0.7948
Balanced Accuracy  : 0.7502

Classification Report:
              precision    recall  f1-score   support

    Eligible     0.8369    0.7569    0.7949     74444
   High_Risk     0.1487    0.6836    0.2442     17488
Not_Eligible     0.9859    0.8101    0.8894    312868

    accuracy                         0.7948    404800
   macro avg     0.6572    0.7502    0.6428    404800
weighted avg     0.9223    0.7948    0.8441    404800



In [20]:
cm = confusion_matrix(
    y_classification,
    classification_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 56345  16145   1954]
 [  3870  11955   1663]
 [  7113  52310 253445]]


In [21]:
X_regression = df.drop(
    columns=[
        "emi_eligibility",
        "max_monthly_emi"
    ]
)

y_regression = df[
    "max_monthly_emi"
]

print("Regression X shape:", X_regression.shape)
print("Regression y shape:", y_regression.shape)

Regression X shape: (404800, 33)
Regression y shape: (404800,)


In [22]:
X_regression_processed = (
    regression_preprocessor.transform(
        X_regression
    )
)

regression_predictions = (
    regression_model.predict(
        X_regression_processed
    )
)

print("Regression predictions generated!")

print(
    "Number of predictions:",
    len(regression_predictions)
)

print(
    "Minimum prediction:",
    regression_predictions.min()
)

print(
    "Maximum prediction:",
    regression_predictions.max()
)

Regression predictions generated!
Number of predictions: 404800
Minimum prediction: -3047.205962335316
Maximum prediction: 84669.20262841442


In [23]:
validation_mae = mean_absolute_error(
    y_regression,
    regression_predictions
)

validation_rmse = np.sqrt(
    mean_squared_error(
        y_regression,
        regression_predictions
    )
)

validation_r2 = r2_score(
    y_regression,
    regression_predictions
)

print("=" * 60)
print("FINAL REGRESSION VALIDATION")
print("=" * 60)

print(
    f"MAE : ₹{validation_mae:,.2f}"
)

print(
    f"RMSE: ₹{validation_rmse:,.2f}"
)

print(
    f"R²  : {validation_r2:.4f}"
)

FINAL REGRESSION VALIDATION
MAE : ₹361.00
RMSE: ₹801.74
R²  : 0.9893


In [24]:
validation_results = pd.DataFrame({
    "Actual_Eligibility": y_classification.values,
    "Predicted_Eligibility": classification_predictions,
    "Actual_Max_EMI": y_regression.values,
    "Predicted_Max_EMI": regression_predictions
})

display(
    validation_results.head(20)
)

,Actual_Eligibility,Predicted_Eligibility,Actual_Max_EMI,Predicted_Max_EMI
0,Not_Eligible,Not_Eligible,500.00,473.731280
1,Not_Eligible,Not_Eligible,700.00,712.642293
2,Eligible,High_Risk,27775.00,27770.305756
3,Eligible,Eligible,16170.00,15508.804748
4,Not_Eligible,Not_Eligible,500.00,664.176681
5,Not_Eligible,Not_Eligible,500.00,648.361062
6,Not_Eligible,Not_Eligible,1950.00,1790.631403
7,Not_Eligible,High_Risk,8260.00,4670.474706
8,Not_Eligible,High_Risk,5500.00,5585.921242
9,Not_Eligible,Not_Eligible,9355.50,9287.194828


In [25]:
validation_results["EMI_Error"] = (
    validation_results["Actual_Max_EMI"]
    - validation_results["Predicted_Max_EMI"]
)

validation_results["Absolute_EMI_Error"] = (
    validation_results["EMI_Error"].abs()
)

display(
    validation_results.head(10)
)

,Actual_Eligibility,Predicted_Eligibility,Actual_Max_EMI,Predicted_Max_EMI,EMI_Error,Absolute_EMI_Error
0,Not_Eligible,Not_Eligible,500.0,473.731280,26.268720,26.268720
1,Not_Eligible,Not_Eligible,700.0,712.642293,-12.642293,12.642293
2,Eligible,High_Risk,27775.0,27770.305756,4.694244,4.694244
3,Eligible,Eligible,16170.0,15508.804748,661.195252,661.195252
4,Not_Eligible,Not_Eligible,500.0,664.176681,-164.176681,164.176681
5,Not_Eligible,Not_Eligible,500.0,648.361062,-148.361062,148.361062
6,Not_Eligible,Not_Eligible,1950.0,1790.631403,159.368597,159.368597
7,Not_Eligible,High_Risk,8260.0,4670.474706,3589.525294,3589.525294
8,Not_Eligible,High_Risk,5500.0,5585.921242,-85.921242,85.921242
9,Not_Eligible,Not_Eligible,9355.5,9287.194828,68.305172,68.305172


In [26]:
VALIDATION_PATH = (
    BASE_PATH
    / "data"
    / "processed"
)

validation_file = (
    VALIDATION_PATH
    / "model_validation_results.csv"
)

validation_results.to_csv(
    validation_file,
    index=False
)

print("Validation results saved successfully!")
print("Path:", validation_file)

Validation results saved successfully!
Path: C:\Users\kalav\OneDrive\Desktop\EMIPredict-AI\data\processed\model_validation_results.csv
